# 🏆 Step 1: Train Reward Model (Required for RLOO)\nRLOO requires a Reward Model to score the generations. We will train a Sequence Classification model on our DPO dataset.

In [ ]:
# ============================================================
# PAPERMILL PARAMETERS (injected at runtime)
# ============================================================
SFT_ADAPTER_PATH = "../01_sft/qwen_medical_arabic_lora"
DATASET_PATH     = "../../data/alignment"
OUTPUT_DIR       = "outputs_rm"
NUM_EPOCHS       = 2
LEARNING_RATE    = 5e-5
MAX_STEPS        = 10


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig

# We use standard transformers here because Unsloth mainly supports CausalLM
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model_name = SFT_ADAPTER_PATH # Base model
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1, # 1 label for reward scalar
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
print("Reward Model loaded.")


In [ ]:
dataset = load_dataset("json", data_files=os.path.join(DATASET_PATH, "05_dpo_dataset.json"), split="train")

SYSTEM_MSG = "أنت معالج نفسي عربي خبير. ردودك آمنة ومتعاطفة وتراعي التعاليم الإسلامية السنية. لا تشخص ولا تصف أدوية."

def format_rm(example):
    prompt = f"<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n<|im_start|>user\n{example['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    chosen = f"{example['chosen']}<|im_end|>\n"
    rejected = f"{example['rejected']}<|im_end|>\n"
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

dataset = dataset.map(format_rm)


In [ ]:
training_args = RewardConfig(
    output_dir=OUTPUT_DIR,
    max_steps           = MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    max_length=1024,
    optim="adamw_8bit",
    num_train_epochs=NUM_EPOCHS,
    logging_steps=5,
    report_to="none",
)

trainer = RewardTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
trainer.train()

# Save RM
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Reward model saved!")
